In [70]:
import sys, os
sys.path.append(os.path.abspath("../src"))   # để import được src/utils.py

import pandas as pd
import numpy as np
from utils import (audit, flag_outliers_iqr, validity_checks,
                   check_consistency, check_duplicates, clean_city_pipeline)
from IPython.display import display

In [71]:
bengaluru_clean = pd.read_csv("../data/interim/bengaluru_after_missing.csv", parse_dates=["Date"])
amritsar_clean  = pd.read_csv("../data/interim/amritsar_after_missing.csv",  parse_dates=["Date"])

print("Bengaluru:", bengaluru_clean.shape, "| Amritsar:", amritsar_clean.shape)
display(bengaluru_clean.head())
display(amritsar_clean.head())

Bengaluru: (2009, 28) | Amritsar: (1221, 29)


,Date,City,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,...,NO2_was_missing,NOx_was_missing,NH3_was_missing,CO_was_missing,SO2_was_missing,O3_was_missing,Benzene_was_missing,Toluene_was_missing,Xylene_was_missing,AQI_was_missing
0,2015-01-01,Bengaluru,NaN,NaN,3.26,17.33,10.88,20.36,0.33,3.54,...,False,False,False,False,False,False,False,False,True,True
1,2015-01-02,Bengaluru,NaN,NaN,6.05,19.73,14.14,23.74,1.35,3.97,...,False,False,False,False,False,False,False,False,True,True
2,2015-01-03,Bengaluru,NaN,NaN,11.91,19.88,20.72,4.32,17.40,13.61,...,False,False,False,False,False,False,False,False,True,True
3,2015-01-04,Bengaluru,NaN,NaN,7.45,21.61,16.88,0.87,5.05,6.52,...,False,False,False,False,False,False,False,False,True,True
4,2015-01-05,Bengaluru,NaN,NaN,9.52,22.17,21.76,31.38,1.83,4.71,...,False,False,False,False,False,False,False,False,True,True


,Date,City,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,...,NO2_was_missing,NOx_was_missing,NH3_was_missing,CO_was_missing,SO2_was_missing,O3_was_missing,Benzene_was_missing,Toluene_was_missing,Xylene_was_missing,AQI_was_missing
0,2017-02-27,Amritsar,13.42,14.92,15.06,8.86,NaN,1.60,0.81,11.00,...,False,True,False,False,False,False,True,True,True,True
1,2017-02-28,Amritsar,12.45,13.80,17.44,8.05,NaN,1.73,0.67,16.06,...,False,True,False,False,False,False,True,True,True,False
2,2017-03-01,Amritsar,19.52,21.88,14.77,6.02,NaN,1.48,0.43,26.24,...,False,True,False,False,False,False,True,True,True,False
3,2017-03-02,Amritsar,77.71,80.27,10.49,3.88,NaN,1.29,0.36,22.35,...,False,True,False,False,False,False,True,True,True,False
4,2017-03-03,Amritsar,19.70,33.08,9.91,3.13,NaN,1.15,0.37,14.29,...,False,True,False,False,False,False,True,True,True,False


In [72]:
# flag_outliers_iqr tính Q1/Q3 RIÊNG cho từng tháng dương lịch (gộp mọi năm),
# nên không gắn nhầm ô nhiễm mùa vụ (đốt rơm rạ, Diwali) thành outlier.
# Hàm CHỈ đánh dấu (thêm cột <col>_is_outlier), KHÔNG xoá/sửa giá trị gốc.
bengaluru_outliers = flag_outliers_iqr(bengaluru_clean)
amritsar_outliers  = flag_outliers_iqr(amritsar_clean)

for name, df in [("BENGALURU", bengaluru_outliers), ("AMRITSAR", amritsar_outliers)]:
    outlier_cols = [c for c in df.columns if c.endswith('_is_outlier')]
    print(f"=== {name}: số ngày bị đánh dấu outlier mỗi cột ===")
    display(df[outlier_cols].sum().sort_values(ascending=False))

=== BENGALURU: số ngày bị đánh dấu outlier mỗi cột ===


CO_is_outlier         290
Toluene_is_outlier    207
Benzene_is_outlier    207
NH3_is_outlier        104
NO_is_outlier          85
PM2.5_is_outlier       81
SO2_is_outlier         70
NO2_is_outlier         46
O3_is_outlier          42
PM10_is_outlier        36
NOx_is_outlier         33
dtype: int64

=== AMRITSAR: số ngày bị đánh dấu outlier mỗi cột ===


NOx_is_outlier        86
NO2_is_outlier        82
NO_is_outlier         81
SO2_is_outlier        72
NH3_is_outlier        69
Toluene_is_outlier    53
PM2.5_is_outlier      34
PM10_is_outlier       34
CO_is_outlier         32
O3_is_outlier         32
Xylene_is_outlier     19
Benzene_is_outlier    18
dtype: int64

In [73]:
print("=== BENGALURU: validity checks ===")
display(validity_checks(bengaluru_outliers))
print("\n=== AMRITSAR: validity checks ===")
display(validity_checks(amritsar_outliers))

=== BENGALURU: validity checks ===


,rule,n_violations
0,PM2.5 không âm,0
1,PM10 không âm,0
2,NO không âm,0
3,NO2 không âm,0
4,NOx không âm,0
5,NH3 không âm,0
6,CO không âm,0
7,SO2 không âm,0
8,O3 không âm,0
9,Benzene không âm,0



=== AMRITSAR: validity checks ===


,rule,n_violations
0,PM2.5 không âm,0
1,PM10 không âm,0
2,NO không âm,0
3,NO2 không âm,0
4,NOx không âm,0
5,NH3 không âm,0
6,CO không âm,0
7,SO2 không âm,0
8,O3 không âm,0
9,Benzene không âm,0


In [74]:
bengaluru_outliers.to_csv("../data/interim/bengaluru_after_outliers.csv", index=False)
amritsar_outliers.to_csv("../data/interim/amritsar_after_outliers.csv", index=False)
print("Đã lưu data/interim/bengaluru_after_outliers.csv và data/interim/amritsar_after_outliers.csv")
print("Bengaluru:", bengaluru_outliers.shape, " | Amritsar:", amritsar_outliers.shape)

Đã lưu data/interim/bengaluru_after_outliers.csv và data/interim/amritsar_after_outliers.csv
Bengaluru: (2009, 39)  | Amritsar: (1221, 41)


In [75]:
for name, df in [("BENGALURU", bengaluru_outliers), ("AMRITSAR", amritsar_outliers)]:
    issues = check_consistency(df)      # kiểm tra tên City và nhãn AQI_Bucket
    n_dup = check_duplicates(df)        # đếm dòng trùng theo (City, Date)
    print(f"=== {name} ===")
    print("Consistency issues:", issues if issues else "Không có vấn đề")
    print("Số dòng trùng (City+Date):", n_dup)
    print()

=== BENGALURU ===
Consistency issues: Không có vấn đề
Số dòng trùng (City+Date): 0

=== AMRITSAR ===
Consistency issues: Không có vấn đề
Số dòng trùng (City+Date): 0



In [76]:
print("=== BENGALURU: audit cuối cùng ===")
display(audit(bengaluru_outliers).sort_values('pct_missing', ascending=False).head(20))
print("\n=== AMRITSAR: audit cuối cùng ===")
display(audit(amritsar_outliers).sort_values('pct_missing', ascending=False).head(20))

=== BENGALURU: audit cuối cùng ===


,dtype,n_missing,pct_missing,n_unique
PM10,float64,318,15.8,1563
NOx,float64,312,15.5,1390
NH3,float64,102,5.1,1448
AQI_Bucket,str,99,4.9,5
AQI,float64,99,4.9,202
Benzene,float64,87,4.3,687
PM2.5,float64,80,4.0,1649
O3,float64,28,1.4,1733
NO,float64,0,0.0,1180
Date,datetime64[us],0,0.0,2009



=== AMRITSAR: audit cuối cùng ===


,dtype,n_missing,pct_missing,n_unique
NOx,float64,255,20.9,838
CO,float64,168,13.8,316
Toluene,float64,119,9.7,515
Xylene,float64,119,9.7,699
Benzene,float64,100,8.2,658
AQI,float64,95,7.8,246
AQI_Bucket,str,95,7.8,6
PM2.5,float64,56,4.6,1091
SO2,float64,51,4.2,824
O3,float64,36,2.9,977


In [77]:
bengaluru_outliers.to_csv("../data/interim/bengaluru_after_w4.csv", index=False)
amritsar_outliers.to_csv("../data/interim/amritsar_after_w4.csv", index=False)
print("Đã lưu data/interim/bengaluru_after_w4.csv và data/interim/amritsar_after_w4.csv")

Đã lưu data/interim/bengaluru_after_w4.csv và data/interim/amritsar_after_w4.csv


In [78]:
# Bengaluru: Xylene thiếu 100% -> drop.
# Amritsar: Xylene chỉ thiếu 17.4% -> GIỮ, nên phải truyền drop_cols=[] rõ ràng
# (nếu để None, hàm sẽ tự dùng DEFAULT_DROP_COLS = ["Xylene"] và drop mất).
# Pipeline tự lưu ra ../data/processed/<city>_clean.csv
bengaluru_final = clean_city_pipeline("Bengaluru", drop_cols=["Xylene"])
print()
amritsar_final = clean_city_pipeline("Amritsar", drop_cols=[])

--- Xử lý Missing Values cho Bengaluru ---
  - NOx: phát hiện 327 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Benzene: phát hiện 64 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Toluene: phát hiện 15 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Đã xoá cột (đề xuất, cần nhóm đồng ý): ['Xylene']
  => Còn 1027 ô missing SAU xử lý (CHỦ Ý giữ NaN ở khối trống dài, không bịa số)
  - Consistency: không có vấn đề
  - Dòng trùng (City+Date): 0
  -> Đã lưu ../data/processed/bengaluru_clean.csv

--- Xử lý Missing Values cho Amritsar ---
  - CO: phát hiện 254 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Toluene: phát hiện 9 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Xylene: phát hiện 9 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  => Còn 1035 ô missing SAU xử lý (CHỦ Ý giữ NaN ở khối trống dài, không bịa số)
  - Consistency: không có vấn đề
  - Dòng trùng (City+Date): 0
  -> Đã lưu ../data/processed/amritsar_clean.csv


In [79]:
print("Bengaluru:", len(bengaluru_final), "dòng,", bengaluru_final.shape[1], "cột")
print("Amritsar:", len(amritsar_final), "dòng,", amritsar_final.shape[1], "cột")

combined_preview = pd.concat([bengaluru_final, amritsar_final], ignore_index=True)
print("Gộp lại:", len(combined_preview), "dòng (phải =",
      len(bengaluru_final) + len(amritsar_final), ")")

only_beng = set(bengaluru_final.columns) - set(amritsar_final.columns)
only_amri = set(amritsar_final.columns) - set(bengaluru_final.columns)
print("\nCột chỉ có ở Bengaluru:", only_beng)
print("Cột chỉ có ở Amritsar:", only_amri)

Bengaluru: 2009 dòng, 39 cột
Amritsar: 1221 dòng, 41 cột
Gộp lại: 3230 dòng (phải = 3230 )

Cột chỉ có ở Bengaluru: set()
Cột chỉ có ở Amritsar: {'Xylene', 'Xylene_is_outlier'}


In [80]:
print(sorted(os.listdir("../data/processed")))

['air_quality_delhi_gurugram_preview.csv', 'amritsar_clean.csv', 'bengaluru_clean.csv', 'delhi_clean.csv', 'gurugram_clean.csv']
